In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm import tqdm
import random

In [2]:
ENTREPOT_PATH = '/home/administrateur/Bureau/Datagrosyst/data_entrepot_outils/'
donnees = {}

def import_df(df_name, path_data, sep, index_col=None):
    donnees[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, index_col=index_col, low_memory=False).replace({'\r\n': '\n'}, regex=True)

def import_dfs(df_names, path_data, sep = ',', index_col=None, verbose=False):
    for df_name in tqdm(df_names) : 
        if(verbose) : 
            print(" - ", df_name)
        import_df(df_name, path_data, sep, index_col=index_col)

tables = [
    'synthetise',
    'sdc',
    'typologie_assol_can_realise',
    'typologie_can_rotation_synthetise',
    'entite_unique_par_sdc_nettoyage',
    'sdc_realise_performance',
    'synthetise_synthetise_performance',
    'intervention_synthetise_agrege',
    'intervention_realise_agrege'
]

# import des données du magasin
import_dfs(tables, ENTREPOT_PATH, sep = ',', verbose=False)

100%|██████████| 9/9 [00:26<00:00,  2.95s/it]


In [3]:
lst_alerte_col = ['alerte_ferti_n_tot', 'alerte_ift_cible_non_mil_chim_tot_hts',
       'alerte_ift_cible_non_mil_f', 'alerte_ift_cible_non_mil_h',
       'alerte_ift_cible_non_mil_i', 'alerte_ift_cible_non_mil_biocontrole',
       'alerte_co_irrigation_std_mil', 'alerte_msn_std_mil_avec_autoconso',
       'alerte_nombre_interventions_phyto', 'alerte_pb_std_mil_avec_autoconso',
       'alerte_rendement', 'alertes_charges', 'alerte_cm_std_mil',
       'alerte_co_semis_std_mil', 'alerte_tps_travail_total']


int_r = donnees['intervention_realise_agrege'][['id','sdc_id']]
sdc_real = donnees['sdc_realise_performance'][['sdc_id']+lst_alerte_col]
sdc = donnees['sdc'][['id','filiere','campagne']].rename(columns={'id':'sdc_id'})
typo_re = donnees['typologie_assol_can_realise'][['sdc_id','typocan_assol_dvlp']]

int_s = donnees['intervention_synthetise_agrege'][['id','synthetise_id','sdc_id']]
synth = donnees['synthetise'][['id']].rename(columns={'id':'synthetise_id'}) # besoin que pour la fin
sdc_synth = donnees['synthetise_synthetise_performance'][['synthetise_id']+lst_alerte_col]
typo_sy = donnees['typologie_can_rotation_synthetise'][['synthetise_id','typocan_rotation']]

unique_entity = donnees['entite_unique_par_sdc_nettoyage']

In [4]:
# Avant tout : Par le jeu des merge on ne garde que les synthé ou sdc qui ont des interventions !
sdc_real = int_r.merge(sdc_real, on='sdc_id', how='left').merge(sdc, on='sdc_id', how='left').merge(typo_re, on='sdc_id', how='left')
sdc_real = sdc_real.groupby('sdc_id').first().reset_index().drop(columns='id')

sdc_synth = int_s.merge(sdc_synth, on='synthetise_id', how='left').merge(sdc, on='sdc_id', how='left').merge(typo_sy, on='synthetise_id', how='left')#.merge(synth, on='synthetise_id', how='left')
sdc_synth = sdc_synth.groupby('synthetise_id').first().reset_index().drop(columns='id')

In [5]:
# Premierement on filtre selon la filière GCPE

sdc_real = sdc_real.loc[sdc_real['filiere'].isin(['POLYCULTURE_ELEVAGE','GRANDES_CULTURES'])]
sdc_synth = sdc_synth.loc[sdc_synth['filiere'].isin(['POLYCULTURE_ELEVAGE','GRANDES_CULTURES'])]

In [6]:
# Deuxiemement on filtre grace à l'outil d'entité unique par sdc
# il regarde dans un sdc s'il y a plusieurs synthétisé et ou zone de réalisé et en choisi un seul

lst_unqiue_entity_real = unique_entity.loc[unique_entity['entite_retenue']=='realise_retenu','sdc_id'].tolist()
sdc_real = sdc_real.loc[sdc_real['sdc_id'].isin(lst_unqiue_entity_real)]

lst_unqiue_entity_synth = unique_entity.loc[unique_entity['entite_retenue']!='realise_retenu','entite_retenue'].to_list()
sdc_synth = sdc_synth.loc[sdc_synth['synthetise_id'].isin(lst_unqiue_entity_synth)]

In [7]:
# Troisiemement on va filtré l'année en cours pour le magasin car les données sont surement en cours de saisie ou en cours de consolidation par les IR
# En gros au 1° avril on accorde lajout de l'année n-1

# Définir les années limites (seuils strictes)
if datetime.now().month <= 3 :
    annees_max = datetime.now().year - 1
else : 
    annees_max = datetime.now().year
    
annees_trop_vieille = 2004 # des pz0 attendues jusqu'en 2005

sdc_synth = sdc_synth.loc[((pd.to_numeric(sdc_synth['campagne'], errors='coerce')) > annees_trop_vieille) & 
                          ((pd.to_numeric(sdc_synth['campagne'], errors='coerce')) < annees_max)]
sdc_real = sdc_real.loc[((pd.to_numeric(sdc_real['campagne'], errors='coerce')) > annees_trop_vieille) & 
                        ((pd.to_numeric(sdc_real['campagne'], errors='coerce')) < annees_max)]

In [8]:
# Quatriemement on filtre par alertes
list_alerte_ok = ["Pas d'alerte", "Cette alerte n'existe pas dans cette filière", "Cette alerte n'existe pas encore dans cette filière"]

def filtrer_alertes(df, lst_alerte_col, list_alerte_ok, name_culture_col):
    # Masque pour les colonnes d'alertes sauf 1
    autres_colonnes = [col for col in lst_alerte_col if col != 'alerte_cm_std_mil']
    mask_autres = df[autres_colonnes].apply(
        lambda x: x.isin(list_alerte_ok) | x.isna()
    ).all(axis=1)

    # Mask pour les alertes de CM lorsque la culture est majoritairement de la prairie
    mask_cm = (
        df['alerte_cm_std_mil'].isin(list_alerte_ok) |
        df['alerte_cm_std_mil'].isna() |
        ((df['alerte_cm_std_mil'].str.contains('<', na=False)) & (df[name_culture_col] == "prairie temporaire >= 50 % assolement"))
        )

    mask_final = mask_autres & mask_cm
    return df[mask_final]


sdc_real = filtrer_alertes(sdc_real, lst_alerte_col, list_alerte_ok, 'typocan_assol_dvlp')
sdc_synth = filtrer_alertes(sdc_synth, lst_alerte_col, list_alerte_ok, 'typocan_rotation')

In [9]:
final_real = sdc[['sdc_id']]
final_real['in_dirodur'] = np.where(final_real['sdc_id'].isin(sdc_real['sdc_id']), True, False)

final_synth = synth[['synthetise_id']]
final_synth['in_dirodur'] = np.where(final_synth['synthetise_id'].isin(sdc_synth['synthetise_id']), True, False)

/tmp/ipykernel_333640/1118103330.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_real['in_dirodur'] = np.where(final_real['sdc_id'].isin(sdc_real['sdc_id']), True, False)


In [10]:
# Au final on ne récupère qu'une liste en synthetise et une liste en réalisé des entités que l'on veut gardé
SORTIE_FILTR_UTIL_REAL = final_real
SORTIE_FILTR_UTIL_SYNTH = final_synth

In [11]:
########## PASSAGE AUX AGREGATIONS #################

In [12]:
tables = [
    'synthetise',
    'sdc',
    'identification_pz0',
    'zone',
    'parcelle'
]

# import des données du magasin
import_dfs(tables, ENTREPOT_PATH, sep = ',', verbose=False)

100%|██████████| 5/5 [00:06<00:00,  1.34s/it]


In [13]:
sdc = donnees['sdc'][['id','code','campagne','modalite_suivi_dephy','code_dephy']].rename(columns={'id':'sdc_id','code':'sdc_code'})
synthetise = donnees['synthetise'][['id','campagnes','sdc_id']].rename(columns={'id':'synthetise_id'})
zone = donnees['zone'][['id','parcelle_id']].rename(columns={'id':'entite_id'})
parcelle = donnees['parcelle'][['id','sdc_id']].rename(columns={'id':'parcelle_id'})
pta = donnees['identification_pz0']

zones_w_pz0 = zone.merge(parcelle, on='parcelle_id', how='left').merge(pta, on='entite_id', how='left')

# SORTIE_FILTR_UTIL_SYNTH, SORTIE_FILTR_UTIL_REAL = FONCTION(donnees)

In [14]:
df_R = SORTIE_FILTR_UTIL_REAL.loc[SORTIE_FILTR_UTIL_REAL['in_dirodur']]\
    .merge(sdc, on='sdc_id', how='left')\
        .drop(columns=['in_dirodur'])

df_S = SORTIE_FILTR_UTIL_SYNTH.loc[SORTIE_FILTR_UTIL_SYNTH['in_dirodur']]\
    .merge(synthetise, on='synthetise_id', how='left')\
        .merge(pta.rename(columns={'entite_id':'synthetise_id'}), on='synthetise_id', how='left')\
            .merge(sdc, on='sdc_id', how='left')\
                .drop(columns=['in_dirodur'])

In [15]:
def list_to_scalar(serie):
    unique_values = list(serie.dropna().unique())
    if len(unique_values) == 0:
        return None
    elif len(unique_values) == 1:
        return unique_values[0]
    else:
        return unique_values

zones_w_pz0 = zones_w_pz0.groupby('sdc_id')['pz0'].apply(lambda x: list_to_scalar(x), include_groups=False).reset_index()

if len(zones_w_pz0.loc[zones_w_pz0['pz0'].apply(lambda x: isinstance(x, list))] ) > 0 :
    raise ValueError("Il y a des sdc réalisé avec plusieurs identification différentes selon leur zones")

df_R = df_R.merge(zones_w_pz0, on='sdc_id', how='left')

In [16]:
df_R['pz0'] = np.where(df_R['modalite_suivi_dephy']=='DETAILLE',
                       df_R['pz0'], 
                       np.where(df_R['modalite_suivi_dephy'].isna(),'non_DEPHY', 'non_suivi'))
df_S['pz0'] = np.where(df_S['modalite_suivi_dephy']=='DETAILLE',
                       df_S['pz0'], 
                       np.where(df_S['modalite_suivi_dephy'].isna(),'non_DEPHY', 'non_suivi'))

In [17]:
df = pd.concat([
    df_S[['sdc_id', 'sdc_code', 'code_dephy', 'campagne', 'pz0', 'synthetise_id', 'campagnes']],
    df_R[['sdc_id', 'sdc_code', 'code_dephy', 'campagne', 'pz0']]
    ])

In [18]:
def extract_years(row):
    years = set()
    if pd.notna(row['campagne']):
        years.add(int(row['campagne']))
    if pd.notna(row['campagnes']):
        years.update(int(y) for y in row['campagnes'].split(', '))
    return sorted(years)

def label_pz0_status(df):
    df_pz0 = df[df['pz0'] == 'pz0'].copy()
    df_pz0['all_years'] = df_pz0.apply(extract_years, axis=1)
    grouped = df_pz0.groupby('code_dephy')['all_years'].agg(lambda x: set().union(*x))
    valid_groups = grouped[grouped.apply(len) >= 2].index.tolist()
    cd_with_pz0_at_the_begging = df.loc[df['pz0'].isin(['pz0','post']),'code_dephy'].tolist()

    df['etat_temporel'] = df.apply(
        lambda row:
            'sans_pz0' if row['code_dephy'] not in cd_with_pz0_at_the_begging and row['code_dephy'] not in valid_groups
            else 'pz0_filtres' if row['code_dephy'] in cd_with_pz0_at_the_begging and row['code_dephy'] not in valid_groups
            else 'pz0' if row['pz0'] == 'pz0' and row['code_dephy'] in valid_groups
            else row['pz0'],
        axis=1
    )

    return df

def find_last_n_year(years, n):
    for i in range(len(years) - n, -1, -1):
        window = years[i:i+n]
        if all(window[j+1] - window[j] == 1 for j in range(len(window)-1)):
            return window
    return None
    
def find_last_consecutive_year_sequence(years):
    if not years:
        return []

    last_3 = find_last_n_year(years, 3) if len(years) >= 3 else None
    last_2 = find_last_n_year(years, 2) if len(years) >= 2 else None

    if last_3 and last_2:
        return last_3 if last_3[-1] >= last_2[-1] else last_2
    return last_3 or last_2 or []

def update_final_status_for_code_dephy_without_point_B(df, codes_with_consecutive):
    for code in list(df['code_dephy'].unique()):
        if code not in codes_with_consecutive:
            mask = df['code_dephy'] == code
            if (df.loc[mask, 'etat_temporel'] == 'sans_pz0').any():
                df.loc[mask, 'etat_temporel'] = 'ni_pz0_ni_point_B'
            elif (df.loc[mask, 'etat_temporel'] == 'pz0_filtres').any():
                df.loc[mask, 'etat_temporel'] = 'pz0_filtres_et_sans_point_B'
            else:
                df.loc[mask, 'etat_temporel'] = 'sans_point_B'

    return df

def get_last_consecutive_years(df):
    df_non_pz0 = df[df['etat_temporel'] != 'pz0'].copy()
    df_non_pz0['all_years'] = df_non_pz0.apply(extract_years, axis=1)
    grouped = df_non_pz0.groupby('code_dephy')['all_years'].agg(lambda x: sorted(set().union(*x)))
    consecutive_years = grouped.apply(find_last_consecutive_year_sequence)

    df = df.set_index('sdc_id')
    df['all_years'] = df.apply(extract_years, axis=1)
    codes_with_consecutive = set()
    for code, years in consecutive_years.items():
        if years:
            codes_with_consecutive.add(code) # garde en mémoire les code ok pour le update final
            mask = (df['code_dephy'] == code) & \
                    ~(df['etat_temporel'].isin(['sans_pz0','pz0_filtres','pz0']))
            for idx, row in df[mask].iterrows():
                if any(y in years for y in row['all_years']):
                    df.loc[idx, 'etat_temporel'] = 'point_B'
    df = df.reset_index()

    df = update_final_status_for_code_dephy_without_point_B(df, codes_with_consecutive)
    return df


def add_etat_temporel_column(df):
    df = label_pz0_status(df)
    df = get_last_consecutive_years(df)
    df['etat_temporel'] = np.where(df['etat_temporel'] == 'post', 'point_I', df['etat_temporel'])
    df.drop(columns=['pz0','all_years'], inplace=True)
    return df.sort_values(['code_dephy','campagne'])

In [19]:
df = add_etat_temporel_column(df)

In [20]:
df.etat_temporel.value_counts()

etat_temporel
pz0_filtres                          2014
point_B                              1519
sans_pz0                             1479
point_I                               931
pz0                                   638
sans_point_B                          379
pz0_filtres_et_sans_point_B           252
incorrect : campagne non-attendue      83
ni_pz0_ni_point_B                      79
incorrect : chevauchement pz0          15
Name: count, dtype: int64

In [23]:
dephy_to_check = random.choices(df.loc[df['etat_temporel']=='pz0_filtres','code_dephy'].dropna().tolist())+\
random.choices(df.loc[df['etat_temporel']=='point_B','code_dephy'].dropna().tolist())+\
random.choices(df.loc[df['etat_temporel']=='sans_pz0','code_dephy'].dropna().tolist())+\
random.choices(df.loc[df['etat_temporel']=='point_I','code_dephy'].dropna().tolist())+\
random.choices(df.loc[df['etat_temporel']=='pz0','code_dephy'].dropna().tolist())+\
random.choices(df.loc[df['etat_temporel']=='sans_point_B','code_dephy'].dropna().tolist())+\
random.choices(df.loc[df['etat_temporel']=='pz0_filtres_et_sans_point_B','code_dephy'].dropna().tolist())+\
random.choices(df.loc[df['etat_temporel']=='incorrect : campagne non-attendue','code_dephy'].dropna().tolist())+\
random.choices(df.loc[df['etat_temporel']=='ni_pz0_ni_point_B','code_dephy'].dropna().tolist())+\
random.choices(df.loc[df['etat_temporel']=='incorrect : chevauchement pz0','code_dephy'].dropna().tolist())

dephy_to_check

['GCF51200',
 'GCF10747',
 'GCF10527',
 'PYF10240',
 'GCF58725',
 'GCF56139',
 'GCF31272',
 'GCF10713',
 'GCF38954',
 'GCF33130']

In [24]:
tables= [
    'synthetise', # util + etat_temp
    'sdc', # util + etat_temp
    'typologie_assol_can_realise', # util
    'typologie_can_rotation_synthetise', # util
    'entite_unique_par_sdc_nettoyage', # util
    'sdc_realise_performance', # util
    'synthetise_synthetise_performance', # util
    'intervention_synthetise_agrege', # util
    'intervention_realise_agrege', # util
    'identification_pz0', # etat_temporel
    'zone', # etat_temporel
    'parcelle' # etat_temporel'
]

donnees = {}
import_dfs(tables, ENTREPOT_PATH, sep = ',', verbose=False)

100%|██████████| 12/12 [00:30<00:00,  2.53s/it]


In [25]:
lst_dephy = random.sample(list(df.loc[df['code_dephy'].notna(),'code_dephy'].unique()),6)

In [26]:
sdc = donnees['sdc']
synthetise = donnees['synthetise']
parcelle = donnees['parcelle']
zone = donnees['zone']
typologie_assol_can_realise = donnees['typologie_assol_can_realise']
typologie_can_rotation_synthetise = donnees['typologie_can_rotation_synthetise']
sdc_realise_performance = donnees['sdc_realise_performance']
intervention_realise_agrege = donnees['intervention_realise_agrege']
synthetise_synthetise_performance = donnees['synthetise_synthetise_performance']
intervention_synthetise_agrege = donnees['intervention_synthetise_agrege']
entite_unique_par_sdc_nettoyage = donnees['entite_unique_par_sdc_nettoyage']
identification_pz0 = donnees['identification_pz0']

In [27]:
sdc = sdc.loc[sdc['code_dephy'].isin(lst_dephy)]
synthetise = synthetise.loc[synthetise['sdc_id'].isin(sdc['id'])]
parcelle = parcelle.loc[parcelle['sdc_id'].isin(sdc['id'])]
zone = zone.loc[zone['parcelle_id'].isin(parcelle['id'])]
typologie_assol_can_realise = typologie_assol_can_realise.loc[typologie_assol_can_realise['sdc_id'].isin(sdc['id'])]
typologie_can_rotation_synthetise = typologie_can_rotation_synthetise.loc[typologie_can_rotation_synthetise['synthetise_id'].isin(synthetise['id'])]
sdc_realise_performance = sdc_realise_performance.loc[sdc_realise_performance['sdc_id'].isin(sdc['id'])]
intervention_realise_agrege = intervention_realise_agrege.loc[intervention_realise_agrege['sdc_id'].isin(sdc['id'])]
synthetise_synthetise_performance = synthetise_synthetise_performance.loc[synthetise_synthetise_performance['synthetise_id'].isin(synthetise['id'])]
intervention_synthetise_agrege = intervention_synthetise_agrege.loc[intervention_synthetise_agrege['synthetise_id'].isin(synthetise['id'])]
entite_unique_par_sdc_nettoyage = entite_unique_par_sdc_nettoyage.loc[entite_unique_par_sdc_nettoyage['sdc_id'].isin(sdc['id'])]
identification_pz0 = identification_pz0.loc[(identification_pz0['entite_id'].isin(synthetise['id'])) | \
                                            (identification_pz0['entite_id'].isin(zone['id']))]

In [28]:
test_path = '/home/administrateur/Bureau/Datagrosyst/catalogue_script_agrosyst/02_outils/tests/data/test_get_temporal_status_for_each_sdc_dirodur/'

sdc.to_csv(test_path +'sdc.csv')
synthetise.to_csv(test_path +'synthetise.csv')
parcelle.to_csv(test_path +'parcelle.csv')
zone.to_csv(test_path +'zone.csv')
typologie_assol_can_realise.to_csv(test_path +'typologie_assol_can_realise.csv')
typologie_can_rotation_synthetise.to_csv(test_path +'typologie_can_rotation_synthetise.csv')
sdc_realise_performance.to_csv(test_path +'sdc_realise_performance.csv')
intervention_realise_agrege.to_csv(test_path +'intervention_realise_agrege.csv')
synthetise_synthetise_performance.to_csv(test_path +'synthetise_synthetise_performance.csv')
intervention_synthetise_agrege.to_csv(test_path +'intervention_synthetise_agrege.csv')
entite_unique_par_sdc_nettoyage.to_csv(test_path +'entite_unique_par_sdc_nettoyage.csv')
identification_pz0.to_csv(test_path +'identification_pz0.csv')

df.loc[df['code_dephy'].isin(dephy_to_check)].to_csv(test_path +'results_test.csv')